<a href="https://colab.research.google.com/github/Rajasbhagat/Google-ADK-Crash-Course/blob/main/ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [2]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [3]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "genai-academy-497020"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'genai-academy-497020' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [5]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [6]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [7]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '8a48ed78-a331-4e11-a7f2-3f31199a78fb'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable:

## A Relaxing & Artsy Day Trip in Sunnyvale, CA

This itinerary focuses on enjoying local art and serene natural spaces without breaking the bank.

---

### Morning (9:00 AM - 12:00 PM): Public Art & Museum Immersion

Start your day with a deep dive into the local art scene. Sunnyvale boasts an extensive public art collection with over 150 artworks scattered throughout the city. You can embark on a self-guided walking tour to discover these pieces, which include sculptures like those from the "Sun Flair" program currently displayed in various Sunnyvale parks until early 2027. The City of Sunnyvale's website offers m

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable:

## A Relaxing & Artsy Day Trip in Sunnyvale, CA

This itinerary focuses on enjoying local art and serene natural spaces without breaking the bank.

---

### Morning (9:00 AM - 12:00 PM): Public Art & Museum Immersion

Start your day with a deep dive into the local art scene. Sunnyvale boasts an extensive public art collection with over 150 artworks scattered throughout the city. You can embark on a self-guided walking tour to discover these pieces, which include sculptures like those from the "Sun Flair" program currently displayed in various Sunnyvale parks until early 2027. The City of Sunnyvale's website offers maps for walking tours, making it easy to explore.

Alternatively, for a more traditional gallery experience, head to nearby Santa Clara:

*   **Triton Museum of Art (Santa Clara):** Enjoy free admission and parking at the Triton Museum of Art. The museum features various exhibitions, with some currently on view through August 2026, such as "Fútbol: The Art of the Game" and "2026 Salon at the Triton Exhibition." The Triton Museum is typically open on Thursdays, but it's always a good idea to check their website for the most current hours.
*   **de Saisset Museum (Santa Clara):** Also offering free admission, the de Saisset Museum on the Santa Clara University campus focuses on Bay Area art and history. Its galleries are usually open Tuesday through Sunday from 11:00 AM to 4:00 PM during the academic year.

---

### Lunch (12:00 PM - 1:30 PM): Affordable Local Bites

Sunnyvale offers a fantastic array of inexpensive and delicious food options. Consider exploring:

*   **Food Trucks:** Look for popular taco trucks like El Califas Taco Truck or Margaritas Taqueria, known for their affordable Mexican cuisine.
*   **Asian Cuisine:** For flavorful and budget-friendly choices, options include Banh Mi from Cam Hung or delicious vegetarian Chinese food from Merit Vegetarian. Madras Cafe is also highlighted for affordable Indian dishes, particularly their dosas.

---

### Afternoon (1:30 PM - 5:00 PM): Relaxing Nature Escape

For a relaxing afternoon, immerse yourself in Sunnyvale's green spaces:

*   **Sunnyvale Baylands Park:** This park combines protected wetlands with developed parkland, offering pathways for biking and hiking, and access to the San Francisco Bay Trail. You can enjoy nature trails and observe wildlife in the 105 acres of seasonal wetlands. While there's a $6 vehicle entry fee from March through October, pedestrian and bicycle entry is free. The park is open from 8:00 AM until 30 minutes after sunset.
*   **Washington Park:** If you prefer a park closer to the city center, Washington Park offers sprawling green fields, ample shade from tall trees, and well-maintained walking paths, making it a serene spot for unwinding.

---

### Evening (5:00 PM onwards): Sunset Views & Casual Exploration

As evening approaches, continue your relaxing and artsy day:

*   **Sunset Stroll & Public Art:** Revisit some public art installations or explore new ones as the light changes. Many public artworks are beautifully illuminated or take on a different character in the evening glow.
*   **Downtown Sunnyvale Ambiance:** Take a leisurely walk around Downtown Sunnyvale, particularly Murphy Avenue. The area offers a pleasant atmosphere for an evening stroll, with various dining options if you choose to have a casual dinner.
*   **Check for Free Events:** Although specific events vary, local platforms like Eventbrite or the City of Sunnyvale's events calendar often list free community activities, concerts, or workshops. For instance, the City sometimes hosts "Sunset Movies Series" in parks during August.

Enjoy your affordable, relaxing, and artsy day trip near Sunnyvale!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [8]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [9]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '3ff0ebe8-2eb1-495d-8fc2-f1320240a23d'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-d0b4ae2e-4fe5-4906-9905-1572fafad0d4',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\x8e\x03\x01\x8f=k_?\xdb\xfa+\x9a,\x92\xec\x19\xb7f\xa7\xa0\x18\x04(\xda+y\xa2H\x0b\xff\x92D\xf3\x1d6an\x80\x02\xf8\x83\xfad9Q\xfd\x97E\xb9\xb5\x18\xdf\x18}\x1f{\x03\xf4d\xb8\x92|v\x916\xfeN.%Y\x85\xcb\x04\xb0\xdf0\x89\x92\xd9\xfaO\xccn\xbd\xf3\xa1tsD"\xb3\xb0\xa03!\xc7...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
    ),
  ],
  prom

The weather near Lake Tahoe is mostly sunny with a high of 66°F and a west wind around 5 mph. These conditions sound perfect for a hike! Enjoy your time out there.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [10]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [11]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '4d1bae64-470a-4130-985e-c2fff6fe2bbf'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Top-rated hotels in San Francisco'
        },
        id='adk-21f76270-fc88-43a4-a093-349149b2f28e',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x97\x03\x01\x8f=k_R\xd7\xd6s?\xef\x0f\xd5\xca*<F1\xa7\x97\x17 a\xaaA\xa37yV\xbc\xd8\xa4\x0b\xf3g\x80\xbft\x06V\x00mD\x14F\xb0}p\xb3e\xba\xc6\xb7\x8b:H\xc9\x8c\n\xaa\x0c\x9f\x8bG\xf7?\x18\x9e\x92|pxD\xdc\xd6FTU\xc0\xbc\xbd\xfe\x8fr\xcby\x83\xba\x00\x0c\xaeT\xe8"...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupt

Certainly! I've found a couple of top-rated hotels in San Francisco:

*   **The Grand Hotel**: 5-star rating with 450 reviews.
*   **Seaside Inn**: 4-star rating with 620 reviews.

The Seaside Inn has the most reviews. For a dinner spot near the Seaside Inn, I recommend **Gary Danko**. Enjoy your evening!

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [12]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [13]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: bbc8e7e1-53c4-4631-936e-be8589105edc

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'bbc8e7e1-53c4-4631-936e-be8589105edc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""That sounds like a fantastic trip! Lisbon is a wonderful city for history and food. I'll help you plan a memorable 2-day itinerary.

Let's start with **Day 1**. How about this for your first day in Lisbon?

### **Day 1: Historic Alfama & Delicious Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the labyrinthine streets of Alfama, Lisbon's oldest district. This historic neighborhood is full of charm, with narrow alleys, traditional houses, and stunning viewpoints.
    *   Visit

That sounds like a fantastic trip! Lisbon is a wonderful city for history and food. I'll help you plan a memorable 2-day itinerary.

Let's start with **Day 1**. How about this for your first day in Lisbon?

### **Day 1: Historic Alfama & Delicious Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the labyrinthine streets of Alfama, Lisbon's oldest district. This historic neighborhood is full of charm, with narrow alleys, traditional houses, and stunning viewpoints.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a majestic Romanesque cathedral that has stood for centuries.
    *   Wander up to the **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking panoramic views over Alfama and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca (tavern) in Alfama, trying dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or *Sardinhas Assadas* (grilled sardines, especially if you're there in summer).
*   **Afternoon (2:30 PM - 6:00 PM): São Jorge Castle**
    *   Ascend to **São Jorge Castle (Castelo de São Jorge)**, a historic Moorish castle offering incredible views, peacocks roaming freely, and a chance to explore ancient fortifications.
*   **Evening (7:00 PM onwards): Dinner in Baixa and Fado Show**
    *   Head to the Baixa district, rebuilt after the 1755 earthquake, for an early dinner.
    *   Consider experiencing a **Fado show** with dinner in Alfama or Bairro Alto, an integral part of Portuguese culture (it's best to book this in advance).

How does this sound for your first day? We can adjust anything you like!

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'bbc8e7e1-53c4-4631-936e-be8589105edc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Apologies, my mistake! You're right, São Jorge Castle was planned for the afternoon. I understood you wanted to replace that. I'm happy to adjust!

Here’s an updated plan for Day 1, replacing the castle visit with another significant historical site:

### **Day 1: Historic Alfama & Carmo Convent**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the labyrinthine streets of Alfama, Lisbon's oldest district. This historic neighborhood is full of charm, with narrow alleys, traditional houses, and stunning viewpoints.
    * 

Apologies, my mistake! You're right, São Jorge Castle was planned for the afternoon. I understood you wanted to replace that. I'm happy to adjust!

Here’s an updated plan for Day 1, replacing the castle visit with another significant historical site:

### **Day 1: Historic Alfama & Carmo Convent**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the labyrinthine streets of Alfama, Lisbon's oldest district. This historic neighborhood is full of charm, with narrow alleys, traditional houses, and stunning viewpoints.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a majestic Romanesque cathedral that has stood for centuries.
    *   Wander up to the **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking panoramic views over Alfama and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal at a local tasca (tavern) in Alfama, trying dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or *Sardinhas Assadas* (grilled sardines, especially if you're there in summer).
*   **Afternoon (2:30 PM - 6:00 PM): Carmo Convent Ruins & Santa Justa Lift**
    *   Head to the **Carmo Convent (Convento do Carmo)**, a fascinating gothic church ruined by the 1755 earthquake, now an open-air archaeological museum. It offers a powerful glimpse into Lisbon's past.
    *   Nearby, you can take a ride on the iconic **Santa Justa Lift** for spectacular views over the Baixa district and beyond (if you're not afraid of heights!).
*   **Evening (7:00 PM onwards): Dinner in Baixa and Fado Show**
    *   Head to the Baixa district, rebuilt after the 1755 earthquake, for an early dinner.
    *   Consider experiencing a **Fado show** with dinner in Alfama or Bairro Alto, an integral part of Portuguese culture (it's best to book this in advance).

How does this revised Day 1 sound?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'bbc8e7e1-53c4-4631-936e-be8589105edc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! I'm glad Day 1 is perfect. Let's move on to **Day 2**, where we'll continue to explore Lisbon's history and indulge in its incredible food scene.

### **Day 2: Belém's Maritime History & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Start your day in the historic district of Belém, known for its monuments related to Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture. This monastery houses the tomb of Vasco da Gama.
    *   

Great! I'm glad Day 1 is perfect. Let's move on to **Day 2**, where we'll continue to explore Lisbon's history and indulge in its incredible food scene.

### **Day 2: Belém's Maritime History & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Start your day in the historic district of Belém, known for its monuments related to Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture. This monastery houses the tomb of Vasco da Gama.
    *   Walk along the Tagus River to see the iconic **Belém Tower (Torre de Belém)**, a fortified tower that served as a ceremonial gateway to Lisbon, and the **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating Portugal's explorers.
*   **Lunch (1:00 PM - 2:30 PM): Original Pastéis de Belém & Seafood Feast**
    *   No visit to Belém is complete without trying the original *Pastéis de Belém* at the historic **Pastéis de Belém** bakery, where the secret recipe has been preserved since 1837.
    *   Follow this sweet treat with a delicious seafood lunch at a local restaurant in the Belém area, perhaps trying *Arroz de Marisco* (seafood rice) or fresh grilled fish.
*   **Afternoon (2:30 PM - 6:00 PM): Time Out Market & Food Exploration**
    *   Head to the **Time Out Market (Mercado da Ribeira)** in Cais do Sodré. This vibrant food hall brings together some of Lisbon's best restaurants and chefs under one roof, offering a fantastic opportunity to sample a wide variety of local and contemporary Portuguese dishes, from gourmet petiscos (tapas) to traditional mains and desserts. It's a true foodie paradise.
*   **Evening (7:00 PM onwards): Dinner in Principe Real or Campo de Ourique**
    *   For your final dinner in Lisbon, explore the charming neighborhoods of Principe Real or Campo de Ourique, known for their excellent and diverse restaurant scenes. You could find anything from modern Portuguese cuisine to cozy, traditional establishments, offering a different culinary experience from your first night.

How does this plan for Day 2 sound to you?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [14]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 06cae2cd-1881-4946-8ef1-8a8ec6245f48
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '06cae2cd-1881-4946-8ef1-8a8ec6245f48'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Olá! Que ótimo! Lisboa é um destino fantástico com uma rica história e comida deliciosa.

Vamos começar com o primeiro dia. Como você está interessado em locais históricos e boa comida local, que tal o seguinte para o seu primeiro dia em Lisboa?

**Dia 1: Lisboa Histórica e Sabores Alfama**

*   **Manhã (9h00 - 13h00): Castelo de São Jorge e Miradouros**
    Comece o dia explorando o icônico Castelo de São Jorge, que oferec

Olá! Que ótimo! Lisboa é um destino fantástico com uma rica história e comida deliciosa.

Vamos começar com o primeiro dia. Como você está interessado em locais históricos e boa comida local, que tal o seguinte para o seu primeiro dia em Lisboa?

**Dia 1: Lisboa Histórica e Sabores Alfama**

*   **Manhã (9h00 - 13h00): Castelo de São Jorge e Miradouros**
    Comece o dia explorando o icônico Castelo de São Jorge, que oferece vistas panorâmicas deslumbrantes da cidade e do rio Tejo. Caminhe pelas muralhas, explore os jardins e descubra a história deste castelo mouro. Depois, passeie pelas ruas estreitas de Alfama e suba até alguns miradouros próximos, como o Miradouro das Portas do Sol ou o Miradouro de Santa Luzia, para mais vistas espetaculares.

*   **Almoço (13h00 - 14h30): Tasca Tradicional em Alfama**
    Em Alfama, você encontrará muitas "tascas" tradicionais, restaurantes pequenos e acolhedores que servem comida portuguesa autêntica. Sugiro experimentar pratos como "Bacalhau à Brás" (bacalhau desfiado com batata palha e ovos) ou "Arroz de Marisco" (arroz de marisco).

*   **Tarde (14h30 - 18h00): Sé de Lisboa e Baixa Pombalina**
    Após o almoço, visite a Sé de Lisboa, a catedral mais antiga da cidade. Em seguida, desça para a Baixa Pombalina, a parte da cidade reconstruída após o terramoto de 1755. Explore as ruas retas e elegantes, a Praça do Comércio e o Arco da Rua Augusta.

*   **Jantar (19h30 em diante): Experiência de Fado com Jantar**
    Para uma autêntica experiência noturna lisboeta, sugiro um jantar com espetáculo de Fado. Há vários restaurantes em Alfama ou no Bairro Alto que oferecem esta combinação, onde pode desfrutar de comida tradicional portuguesa enquanto ouve a melancólica e bela música de Fado, Património Imaterial da Humanidade.

O que achas deste plano para o teu primeiro dia?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 40ee4e70-ead5-4d75-91a9-ef664030a05d
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '40ee4e70-ead5-4d75-91a9-ef664030a05d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! Let's plan Day 2 in Paris, keeping the cultural and iconic landmarks theme in mind.

### Day 2: Artistic Montmartre & Historic Île de la Cité

*   **Morning (9:00 AM - 1:00 PM): Île de la Cité and Sainte-Chapelle**
    *   Start your day by exploring the historic heart of Paris, the Île de la Cité. Walk around the exterior of the magnificent Notre Dame Cathedral (currently under restoration, but still impressive from the outside).
    *   Visit the stunning Sainte-Chapelle, renowned for its breathtaking stained-glass windows depicting biblical scenes.
    *   Stroll through the charming

Okay, fantastic! Let's plan Day 2 in Paris, keeping the cultural and iconic landmarks theme in mind.

### Day 2: Artistic Montmartre & Historic Île de la Cité

*   **Morning (9:00 AM - 1:00 PM): Île de la Cité and Sainte-Chapelle**
    *   Start your day by exploring the historic heart of Paris, the Île de la Cité. Walk around the exterior of the magnificent Notre Dame Cathedral (currently under restoration, but still impressive from the outside).
    *   Visit the stunning Sainte-Chapelle, renowned for its breathtaking stained-glass windows depicting biblical scenes.
    *   Stroll through the charming flower market (Marché aux Fleurs Reine Elizabeth II) nearby.
*   **Lunch (1:00 PM - 2:00 PM): Local Bistro near Pont Neuf**
    *   Enjoy a traditional French lunch at a cozy bistro on Île de la Cité or near Pont Neuf, perhaps trying a classic Croque Monsieur or a savory crêpe.
*   **Afternoon (2:30 PM - 6:30 PM): Montmartre and Sacré-Cœur**
    *   Head to the charming and bohemian Montmartre district. Take the funicular or walk up to the iconic Sacré-Cœur Basilica for panoramic views of Paris.
    *   Explore the lively Place du Tertre, where artists set up their easels and paint portraits and caricatures.
    *   Wander through the picturesque streets, discover hidden vineyards (Clos Montmartre), and soak in the artistic atmosphere.
*   **Evening (7:30 PM onwards): Dinner & Parisian Cabaret (Optional)**
    *   Enjoy a delicious dinner in Montmartre, choosing from its wide array of restaurants, from traditional French to more modern cuisines.
    *   For a quintessential Parisian experience, consider an optional evening show at the famous Moulin Rouge or another cabaret (booking in advance is highly recommended).

How does this sound for your second day?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
